# Parallel kinetic dynamics with open, nonperiodic boundaries

We evolve a collisionless distribution in one spatial and one velocity dimension:
$$\partial_t f+v_\parallel\partial_zf+a\partial_{v_\parallel}f=0,
\qquad a=qE_\parallel/m=0.3.$$
The finite phase-space domain is $0\le z\le1$, $-2.5\le v_\parallel\le2.5$,
and $0\le t\le1.2$. Units are normalized; particle mass is 1.
The electric field is **prescribed and constant**. This benchmark isolates
parallel streaming and acceleration; it is not self-consistent Vlasov–Poisson
or full gyrokinetics, and contains no collision or mirror term.

Two unequal warm beam pulses enter from opposite ends, interpenetrate without
collisions, accelerate, and leave. Boundary data are imposed **only on inflow**:

| Face | Prescribed incoming characteristics | Outgoing characteristics |
| --- | --- | --- |
| $z=0$ | $v>0$: left reservoir | $v<0$: leave freely |
| $z=1$ | $v<0$: right reservoir | $v>0$: leave freely |
| $v=-2.5$ | incoming because $a>0$ | none |
| $v=2.5$ | none | outgoing because $a>0$ |

No endpoints are identified or wrapped. Velocity-boundary fluxes are included
in the diagnostics even when small. Install `python -m pip install -e '.[host,notebook,test]' -e './packages/models[precision,test]'` from the repository root and select that kernel.


In [ ]:
import bspf_models.kinetic.parallel_kinetic as bspf_parallel_kinetic
import pybspf.calculus as bspf_calculus
import pybspf.plans as bspf_plans

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import pybspf as b

acceleration = 0.3
times = jnp.linspace(0., 1.2, 121)

def reference(t, z, v):
    z0 = z-v*t+0.5*acceleration*t*t
    v0 = v-acceleration*t
    left = jnp.exp(-((z0+0.2)/0.18)**2-((v0-1.0)/0.32)**2)
    right = 0.7*jnp.exp(-((z0-1.2)/0.20)**2-((v0+0.8)/0.35)**2)
    return left+right

inflow = reference


## Initial state and reservoirs with an independent reference

Let $F(z,v)$ be the sum of the two Gaussian pulses specified above, centered
initially at $(-0.2,1)$ and $(1.2,-0.8)$ in the extended phase space. Their
restrictions give the initial interior state and incoming reservoir traces.
The exact constant-acceleration solution is
$$f_*(z,v,t)=F(z-vt+\tfrac12at^2,v-at).$$
This choice makes the physically open boundary experiment directly verifiable.
Only initial values and incoming traces are supplied to the solver; the interior
and outgoing values are evolved numerically. Changing `inflow` allows other
reservoir distributions without changing the transport algorithm.

The reusable JAX solver uses a tensor BSPF weak form with resolved quadrature
and upwind boundary fluxes. Inflow is imposed weakly, so its actual trace error
is measured. RK4 has an advection stability restriction; this example uses
$\Delta t=0.0005$. There is no positivity limiter, clipping or filtering.


In [ ]:
def solve(nz=65, nv=129, substeps=20, quadrature_order=8):
    z, v = jnp.linspace(0., 1., nz), jnp.linspace(-2.5, 2.5, nv)
    zp = bspf_plans.plan_1d(z, degree=7, n_basis=16, boundary_points=9)
    vp = bspf_plans.plan_1d(v, degree=7, n_basis=24, boundary_points=9)
    model = bspf_parallel_kinetic.plan_parallel_kinetic(zp, vp, acceleration=acceleration,
                                    quadrature_order=quadrature_order)
    initial = reference(0., z[:, None], v[None, :])
    f = bspf_parallel_kinetic.integrate_parallel_kinetic(model, initial, times,
                                      inflow=inflow, substeps=substeps)
    return z, v, zp, vp, f

z, v, zp, vp, f = solve()
exact = reference(times[:, None, None], z[None, :, None], v[None, None, :])
error = jnp.max(jnp.abs(f-exact), axis=(1, 2))
field_error = float(jnp.max(error))
inflow_error = float(jnp.max(jnp.concatenate((
    jnp.abs(f[:, 0, v > 0]-exact[:, 0, v > 0]).ravel(),
    jnp.abs(f[:, -1, v < 0]-exact[:, -1, v < 0]).ravel(),
    jnp.abs(f[:, :, 0]-exact[:, :, 0]).ravel()))))
minimum = float(jnp.min(f))
endpoint_mismatch = float(jnp.max(jnp.abs(exact[:, 0]-exact[:, -1])))
print(f"Maximum distribution error: {field_error:.3e}")
print(f"Incoming trace error:       {inflow_error:.3e}")
print(f"Minimum f (no clipping):    {minimum:.3e}")
print(f"Opposite-end mismatch:      {endpoint_mismatch:.3e}")
assert field_error < 2e-7 and inflow_error < 2e-7
assert minimum > -1e-7 and endpoint_mismatch > 0.1


## Resolution and time-step checks

Compare a 33×65 grid against 65×129 using the independent characteristic
solution, then halve the time step and increase Gauss order independently.
Negative undershoots are reported explicitly; the method has no general
positivity guarantee and these tolerances concern this smooth benchmark only.


In [ ]:
zc, vc, _, _, fc = solve(33, 65)
coarse_error = float(jnp.max(jnp.abs(fc-reference(
    times[:, None, None], zc[None, :, None], vc[None, None, :]))))
_, _, _, _, half_step = solve(substeps=40)
_, _, _, _, higher_quad = solve(quadrature_order=10)
time_change = float(jnp.max(jnp.abs(half_step-f)))
quadrature_change = float(jnp.max(jnp.abs(higher_quad-f)))
print(f"33x65 reference error:      {coarse_error:.3e}")
print(f"Halved-step change:         {time_change:.3e}")
print(f"Gauss 8 -> 10 change:       {quadrature_change:.3e}")
assert coarse_error > 100*field_error
assert time_change < 1e-7 and quadrature_change < 1e-7


## Particle and kinetic-energy balances using BSPF integration

All moments use BSPF spatial/velocity integration; all accumulated fluxes use
BSPF time antiderivatives. On this **finite** phase-space rectangle,
$$N'=\int v(f_L-f_R)\,dv+a\int(f_{v_-}-f_{v_+})\,dz,$$
$$K'=a\iint vf\,dz\,dv+\tfrac12\int v^3(f_L-f_R)\,dv
+\tfrac a2\int(v_-^2f_{v_-}-v_+^2f_{v_+})\,dz.$$
Here $K=\frac12\iint v^2f\,dz\,dv$. Particle number changes through the
boundaries; kinetic energy also changes through electric work. Physical fluxes
are evaluated from the computed traces, not fitted to moment differences.
The residual therefore includes weak boundary-trace and integration errors.


In [ ]:
def moment(field):
    return bspf_calculus.integrate(zp, bspf_calculus.integrate(vp, jnp.transpose(field, (2, 1, 0))))

number = moment(f)
momentum = moment(f*v)
kinetic_energy = moment(0.5*f*v**2)
density = bspf_calculus.integrate(vp, jnp.transpose(f, (2, 0, 1)))
particle_flux = (bspf_calculus.integrate(vp, ((f[:, 0]-f[:, -1])*v).T)
                 +acceleration*bspf_calculus.integrate(zp, (f[:, :, 0]-f[:, :, -1]).T))
energy_flux = (bspf_calculus.integrate(vp, (0.5*(f[:, 0]-f[:, -1])*v**3).T)
    +0.5*acceleration*bspf_calculus.integrate(zp,
        (v[0]**2*f[:, :, 0]-v[-1]**2*f[:, :, -1]).T))
time_plan = bspf_plans.plan_1d(times, degree=7, n_basis=24, boundary_points=9)
number_from_flux = number[0]+bspf_calculus.antiderivative(time_plan, particle_flux)
energy_from_work = kinetic_energy[0]+bspf_calculus.antiderivative(
    time_plan, acceleration*momentum+energy_flux)
number_error = float(jnp.max(jnp.abs(number-moment(exact)))/jnp.max(moment(exact)))
number_balance = float(jnp.max(jnp.abs(number-number_from_flux))/jnp.max(number))
energy_balance = float(jnp.max(jnp.abs(kinetic_energy-energy_from_work))/jnp.max(kinetic_energy))
print(f"Relative particle reference error: {number_error:.3e}")
print(f"Relative particle-balance error:   {number_balance:.3e}")
print(f"Relative energy-balance error:     {energy_balance:.3e}")
assert number_error < 1e-6
assert number_balance < 2e-5 and energy_balance < 2e-5


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
for ax, index in zip(axes[0], (0, 50, 120)):
    image = ax.pcolormesh(z, v, f[index].T, shading="auto", cmap="magma", vmin=0, vmax=1)
    ax.set(xlabel="z", ylabel="Parallel velocity", title=f"Distribution at t={float(times[index]):g}")
fig.colorbar(image, ax=list(axes[0]), label="f (tiny negative values reported above)")
for index in (0, 25, 50, 80, 120):
    axes[1, 0].plot(z, density[index], label=f"t={float(times[index]):g}")
axes[1, 0].set(xlabel="z", ylabel="Number density", title="Open-boundary injection and escape")
axes[1, 0].legend()
axes[1, 1].plot(times, number, label="BSPF particle integral")
axes[1, 1].plot(times, number_from_flux, "--", label="Initial + boundary transfer")
axes[1, 1].set(xlabel="t", ylabel="Particles", title="Finite-domain particle balance")
axes[1, 1].legend()
axes[1, 2].semilogy(times[1:], error[1:])
axes[1, 2].set(xlabel="t", ylabel="Maximum distribution error", title="Independent characteristic reference")
plt.show()
